# 위성 시계열 데이터 기반 Semi-Supervised Learning 실험

세 논문(CGMatch, InterLUDE, SC-SSL)이 각자 완화한 가정의 위배 조건에서 고전 기법(Self-training)보다 실제로 더 기여하는지 본다.

- 방법 6종 × 조건 3종 × seed 3개 = 54회 학습
- 로직은 `sslexp/*.py`에 있고 이 노트북은 호출만 한다. 설명은 `README.md`.
- 끊기면 같은 셀을 다시 실행한다(끝난 실행은 건너뛴다).

In [1]:
# 0. 준비: Drive 연결, 경로, 코드 불러오기
from google.colab import drive
drive.mount('/content/drive')
import os, sys, importlib, torch

DATA     = "/content/drive/MyDrive/AI4Biodynamics/data"            # 읽기만 한다
CODE_DIR = "/content/drive/MyDrive/ssl_seminar/ssl_experiment"     # sslexp 폴더가 바로 아래에 있는 위치
OUT      = "/content/drive/MyDrive/ssl_seminar/results_v1"         # 설정을 바꾸면 새 폴더로

if not os.path.isdir(f"{CODE_DIR}/sslexp"):
    found = [r for r, d, _ in os.walk("/content/drive/MyDrive") if "sslexp" in d][:5]
    raise FileNotFoundError(f"{CODE_DIR} 아래에 sslexp 폴더가 없습니다. CODE_DIR을 고치세요. Drive에서 찾은 위치: {found if found else '없음(폴더를 아직 올리지 않았습니다)'}")
sys.path.insert(0, CODE_DIR)
import sslexp.data as D, sslexp.methods as M, sslexp.run as R
D, M, R = importlib.reload(D), importlib.reload(M), importlib.reload(R)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(런타임 유형을 GPU로 바꾸세요)")

Mounted at /content/drive
GPU: Tesla T4


In [2]:
# 1. 데이터 준비와 규모 확인 (학습을 시작하기 전에 이 출력을 본다)
MIN_TRAIN = 300      # 학습 구역 지점이 이보다 적은 클래스는 뺀다(100보다 커야 한다)
bundle = R.prepare(csv=f"{DATA}/HLS_TimeSeries_2022.csv", dense_cache=f"{DATA}/hls_dense_cache.pt", xy_cache=f"{DATA}/site_xy.npy",
                   work_dir=OUT, split="block", block_km=25.0, min_train=MIN_TRAIN)
print()
D.describe_conditions(bundle, ("base", "A", "B"))

[데이터] 지점 24,009 | 클래스 24 -> 사용 24 | 제외 없음
[분할]   train 14,359 / val 4,799 / test 4,851 | 블록 253개(25.0 km)
         원래 클래스: (train, val, test) 지점 수
          0: (641, 164, 195)
          1: (463, 295, 242)
          2: (641, 176, 183)
          3: (650, 161, 189)
          4: (641, 178, 181)
          5: (693, 200, 107)
          6: (647, 156, 197)
          7: (482, 260, 258)
          8: (722, 142, 136)
          9: (637, 189, 174)
         10: (641, 162, 197)
         11: (597, 195, 208)
         12: (538, 173, 289)
         13: (525, 195, 280)
         14: (572, 295, 133)
         15: (686, 169, 148)
         16: (561, 158, 281)
         17: (562, 248, 190)
         18: (499, 274, 227)
         19: (754, 100, 146)
         20: (557, 252, 191)
         21: (385, 387, 228)
         22: (786, 101, 113)
         23: (479, 169, 358)

[조건 base] 기준 조건(가정 성립) | labeled 2,400 (클래스당 100~100) | unlabeled 6,840 (클래스당 285~285)
[조건 A] 극소량 레이블 | labeled 96 (클래스당 4~4) | unlabeled 6,840 (클래스당 285~28

In [3]:
# (선택) 학습을 시작하지 않고 예상 소요 시간만 본다. 끝난 실행이 있으면 그 기록으로, 없으면 20스텝 시험 학습으로 계산한다.
_ = R.estimate_time(bundle, OUT, cfg=M.Cfg(iters=6000))

[예상 소요 시간] 남은 54회 / 전체 54회 -> 약 5시간 52분  (시험 학습 20스텝 기준, device cuda)
                 1회당: Supervised only 33초, Self-training 4분, FixMatch 8분, CGMatch 8분, InterLUDE 8분, SC-SSL 8분
                 평가·저장 시간은 빠져 있어 실제로는 조금 더 걸린다. 실행이 하나 끝날 때마다 다시 계산한다.


In [4]:
# 2. 주 실험. 시작할 때 예상 소요 시간이 출력되고, 도는 동안 진행 막대 두 줄이 보인다.
#    위 막대 = 전체 진행(경과 시간 · 남은 시간), 아래 막대 = 지금 도는 실행(스텝 · val F1)
#    실행이 하나 끝날 때마다 결과 한 줄과 다시 계산한 남은 예상 시간이 찍힌다.
#    처음에는 아래 두 줄을 SEEDS = [0], CONDITIONS = ["A"]로 바꿔 한 묶음만 돌려 본다.
SEEDS      = [0, 1, 2]
CONDITIONS = ["base", "A", "B"]
METHODS    = list(M.METHODS)          # supervised, pseudolabel, fixmatch, cgmatch, interlude, scssl
cfg = M.Cfg(iters=6000)

R.run_grid(bundle, OUT, methods=METHODS, conditions=CONDITIONS, seeds=SEEDS, cfg=cfg)

[실행] 전체 54회 중 남은 것 54회 | device cuda | iters 6000
[예상 소요 시간] 남은 54회 / 전체 54회 -> 약 5시간 53분  (시험 학습 20스텝 기준, device cuda)
                 1회당: Supervised only 33초, Self-training 4분, FixMatch 8분, CGMatch 8분, InterLUDE 8분, SC-SSL 8분
                 평가·저장 시간은 빠져 있어 실제로는 조금 더 걸린다. 실행이 하나 끝날 때마다 다시 계산한다.


전체 0/54회   0%|          | 경과 00:00 · 남은 시간 ?

  base · Supervised only · seed0   0%|          | 0/6000 스텝 · 남은 시간 ?

  [1/54] base Supervised only seed0 | test macro-F1 0.644 acc 0.645 (best it 3750) | 1분, 누적 1분, 남은 예상 19시간 13분


  base · Self-training · seed0   0%|          | 0/6000 스텝 · 남은 시간 ?

  [2/54] base Self-training   seed0 | test macro-F1 0.641 acc 0.647 (best it 3250) | 5분, 누적 7분, 남은 예상 8시간 09분


... (100 outputs left out)

  B · InterLUDE · seed2   0%|          | 0/6000 스텝 · 남은 시간 ?

  [53/54]    B InterLUDE       seed2 | test macro-F1 0.488 acc 0.548 (best it 3250) | 10분, 누적 6시간 37분, 남은 예상 9분


  B · SC-SSL · seed2   0%|          | 0/6000 스텝 · 남은 시간 ?

  [54/54]    B SC-SSL          seed2 | test macro-F1 0.567 acc 0.586 (best it 4250) | 9분, 누적 6시간 46분, 남은 예상 0초
[완료] 54회, 총 6시간 46분


In [5]:
# 3. 결과 표 (OUT/tables 에도 저장된다)
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
T = R.tables(OUT)
for name, t in T.items():
    print(f"\n== {name}"); display(t)


== test_macro_f1


,기준 조건,조건 A (극소량 레이블),조건 B (분포 불일치)
Supervised only,0.659 ± 0.007,0.407 ± 0.013,0.506 ± 0.017
Self-training,0.652 ± 0.010,0.402 ± 0.013,0.498 ± 0.026
FixMatch,0.654 ± 0.004,0.441 ± 0.011,0.507 ± 0.028
CGMatch,0.654 ± 0.007,0.433 ± 0.028,0.500 ± 0.023
InterLUDE,0.658 ± 0.007,0.428 ± 0.038,0.493 ± 0.005
SC-SSL,0.644 ± 0.004,0.435 ± 0.036,0.555 ± 0.029



== gain_vs_fixmatch


,기준 조건,조건 A (극소량 레이블),조건 B (분포 불일치)
CGMatch,+0.000 ± 0.010,-0.008 ± 0.017,-0.006 ± 0.005
InterLUDE,+0.004 ± 0.007,-0.014 ± 0.030,-0.014 ± 0.032
SC-SSL,-0.009 ± 0.005,-0.007 ± 0.025,+0.049 ± 0.021



== gain_vs_selftraining


,기준 조건,조건 A (극소량 레이블),조건 B (분포 불일치)
FixMatch,+0.002 ± 0.006,+0.040 ± 0.014,+0.009 ± 0.011
CGMatch,+0.002 ± 0.016,+0.031 ± 0.027,+0.002 ± 0.011
InterLUDE,+0.006 ± 0.009,+0.026 ± 0.028,-0.005 ± 0.028
SC-SSL,-0.007 ± 0.011,+0.033 ± 0.035,+0.057 ± 0.011



== tail_f1


,조건 B (분포 불일치)
Supervised only,0.429 ± 0.034
Self-training,0.419 ± 0.024
FixMatch,0.429 ± 0.047
CGMatch,0.417 ± 0.036
InterLUDE,0.396 ± 0.012
SC-SSL,0.512 ± 0.046



== diagnosis


,cond,method,pseudo-label 활용률,pseudo-label 정확도,unlabeled 정확도(학습 구역),test 정확도,강한 증강 전후 예측 변화율,분/회
0,A,CGMatch,0.589,0.361,0.509,0.448,NaN,8.991
1,A,FixMatch,0.771,0.548,0.503,0.455,NaN,9.464
2,A,InterLUDE,0.546,0.613,0.493,0.439,NaN,10.150
3,A,Self-training,0.642,0.562,0.473,0.410,NaN,5.227
4,A,SC-SSL,0.527,0.611,0.497,0.441,NaN,9.568
5,A,Supervised only,0.000,0.000,0.479,0.417,0.562,1.747
6,B,CGMatch,0.937,0.448,0.464,0.551,NaN,8.974
7,B,FixMatch,0.642,0.516,0.479,0.555,NaN,9.473
8,B,InterLUDE,0.417,0.441,0.413,0.556,NaN,10.147
9,B,Self-training,0.674,0.474,0.464,0.547,NaN,5.234


## 선택: 조건 Bc (B의 대조군)

조건 B는 기준 조건과 두 가지가 다르다(labeled가 long-tail, unlabeled 분포가 반대). `Bc`는 labeled는 B와 같고 unlabeled만 같은 방향으로 둔 조건이라, B와 Bc를 비교하면 "분포 불일치"만의 영향을 볼 수 있다. 18회 학습이 추가된다.

In [6]:
# R.run_grid(bundle, OUT, methods=METHODS, conditions=["Bc"], seeds=SEEDS, cfg=cfg)
# for name, t in R.tables(OUT).items(): print(f"\n== {name}"); display(t)